# 01B — Lead Scoring Pipeline Diagnostic
## bd_replica_crm · localizar el cuello de botella `evidence → model → score`

Este notebook está diseñado para responder una sola pregunta:

> **¿En qué etapa exacta del pipeline se está consumiendo el tiempo y por qué todavía tenemos `lead_scores = 0`?**

Por defecto es **diagnóstico y no destructivo**. No entrena, no promueve y no escribe scores salvo que actives explícitamente los flags del final.

Mide:

- conectividad PostgreSQL;
- volumen y madurez de `features.lead_evidence`;
- tiempo de lectura SQL;
- memoria estimada de los datasets;
- cardinalidad de categóricas;
- split temporal;
- tiempo de carga de cada target;
- costo de la query de scoring live;
- existencia de `model_runs`, aliases y artifacts;
- locks / queries activas;
- diagnóstico de índices;
- posible cuello de botella en `refresh_historical_features`;
- posible cuello de botella en `executemany` para `lead_scores`;
- resumen ejecutivo con ranking de problemas.


## 0. Ejecución

```powershell
cd C:\Users\user\Documents\dwh\bd_replica_crm
.\.venv\Scripts\Activate.ps1
pip install -e .
```

Selecciona `.venv` como kernel en VS Code y ejecuta **Run All**.


In [ ]:
from __future__ import annotations

import sys
import os
import gc
import json
import time
import math
import traceback
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Ejecuta este notebook dentro de bd_replica_crm.")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from replica_cygnus.settings import load_settings
from replica_cygnus.connections import connect_postgres
from replica_cygnus.lead_scoring.config import load_lead_scoring_config
from replica_cygnus.lead_scoring.training import (
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    MODEL_FEATURES,
    temporal_split,
)
from replica_cygnus.lead_scoring.scoring import _read_scoring_frame
from replica_cygnus.lead_scoring.registry import serving_model

settings = load_settings(PROJECT_ROOT)

config_path = PROJECT_ROOT / "config" / "lead_scoring.yml"
if not config_path.exists():
    config_path = PROJECT_ROOT / "config" / "lead_scoring.example.yml"

if not config_path.exists():
    raise FileNotFoundError(
        "No encuentro config/lead_scoring.yml ni config/lead_scoring.example.yml"
    )

cfg = load_lead_scoring_config(config_path)
conn = connect_postgres(settings)

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 150)
pd.set_option("display.width", 220)

TIMINGS = []

def timed(name, fn):
    t0 = time.perf_counter()
    status = "OK"
    detail = None
    try:
        result = fn()
        return result
    except Exception as exc:
        status = "ERROR"
        detail = repr(exc)
        raise
    finally:
        elapsed = time.perf_counter() - t0
        TIMINGS.append({
            "stage": name,
            "seconds": elapsed,
            "status": status,
            "detail": detail,
        })
        print(f"[{status}] {name}: {elapsed:.3f}s")

def df(sql, params=None):
    t0 = time.perf_counter()
    out = pd.read_sql_query(sql, conn, params=params)
    elapsed = time.perf_counter() - t0
    TIMINGS.append({
        "stage": "SQL",
        "seconds": elapsed,
        "status": "OK",
        "detail": sql.strip().splitlines()[0][:100],
    })
    return out

def mem_mb(frame):
    return frame.memory_usage(deep=True).sum() / (1024**2)

print("Repo:", PROJECT_ROOT)
print("Database:", settings.postgres.database)
print("score_window_days:", cfg.score_window_days)
print("training_min_rows:", cfg.training_min_rows)
print("validation_days:", cfg.validation_days)
print("test_days:", cfg.test_days)


## 1. Estado actual del pipeline


In [ ]:
status = df('''
SELECT
  (SELECT COUNT(*) FROM features.lead_evidence) AS evidence_rows,
  (SELECT COUNT(*) FROM features.lead_evidence WHERE evidence_source='LIVE') AS live_rows,
  (SELECT COUNT(*) FROM features.lead_evidence WHERE separacion_14d IS NOT NULL) AS sep_matured,
  (SELECT COUNT(*) FROM features.lead_evidence WHERE minuta_60d IS NOT NULL) AS minuta_matured,
  (SELECT COUNT(*) FROM decision_intelligence.lead_scores) AS scores,
  (SELECT COUNT(*) FROM decision_intelligence.recommendations WHERE decision_system='priorizacion_leads') AS recommendations,
  (SELECT COUNT(*) FROM decision_intelligence.actions) AS actions,
  (SELECT COUNT(*) FROM decision_intelligence.outcomes WHERE decision_system='priorizacion_leads') AS outcomes,
  (SELECT COUNT(*) FROM model_control.model_runs WHERE decision_system='priorizacion_leads') AS model_runs,
  (SELECT COUNT(*) FROM model_control.model_aliases WHERE decision_system='priorizacion_leads') AS aliases
''')

status.T.rename(columns={0:"value"})


## 2. Queries activas y locks


In [ ]:
activity = df('''
SELECT
    pid,
    usename,
    state,
    wait_event_type,
    wait_event,
    now() - query_start AS running_for,
    LEFT(query, 300) AS query
FROM pg_stat_activity
WHERE datname = current_database()
  AND pid <> pg_backend_pid()
ORDER BY query_start
''')

activity


In [ ]:
locks = df('''
SELECT
    a.pid,
    a.usename,
    a.state,
    l.locktype,
    l.mode,
    l.granted,
    c.relname,
    now() - a.query_start AS running_for,
    LEFT(a.query, 200) AS query
FROM pg_locks l
JOIN pg_stat_activity a ON a.pid = l.pid
LEFT JOIN pg_class c ON c.oid = l.relation
WHERE a.datname = current_database()
ORDER BY l.granted, a.query_start
''')

locks.head(100)


## 3. Índices relevantes


In [ ]:
indexes = df('''
SELECT
    schemaname,
    tablename,
    indexname,
    indexdef
FROM pg_indexes
WHERE
    (schemaname='features' AND tablename='lead_evidence')
 OR (schemaname='decision_intelligence' AND tablename='lead_scores')
 OR (schemaname='core' AND tablename='fact_ciclo_comercial_unidad')
ORDER BY schemaname, tablename, indexname
''')

indexes


La query de scoring live filtra por:

- `features.lead_evidence.decision_at`
- `features_refreshed_at IS NOT NULL`
- `documento_cliente`
- `codigo_proyecto`
- y un `NOT EXISTS` contra `core.fact_ciclo_comercial_unidad`

Si faltan índices adecuados en `core.fact_ciclo_comercial_unidad`, esta etapa puede ser el principal cuello de botella.


## 4. Volumen de evidencia y cardinalidad


In [ ]:
evidence_summary = timed(
    "evidence_summary",
    lambda: df('''
        SELECT
            COUNT(*) AS rows,
            COUNT(DISTINCT evidence_key) AS unique_evidence,
            COUNT(DISTINCT lead_id) AS unique_leads,
            COUNT(DISTINCT codigo_proyecto) AS projects,
            COUNT(DISTINCT asesor) AS advisors,
            COUNT(DISTINCT canal) AS channels,
            COUNT(DISTINCT medio) AS media,
            MIN(decision_at) AS min_decision_at,
            MAX(decision_at) AS max_decision_at,
            MAX(features_refreshed_at) AS max_features_refreshed_at
        FROM features.lead_evidence
    ''')
)

evidence_summary.T


## 5. Tiempo de lectura: target separación


In [ ]:
sep_columns = ", ".join(
    ["evidence_key","decision_at",*MODEL_FEATURES,"separacion_14d"]
)

def load_sep():
    return df(f'''
        SELECT {sep_columns}
        FROM features.lead_evidence
        WHERE separacion_14d IS NOT NULL
          AND features_refreshed_at IS NOT NULL
        ORDER BY decision_at, evidence_key
    ''')

sep_frame = timed("load_sep_frame", load_sep)

print("rows:", len(sep_frame))
print("memory_mb:", round(mem_mb(sep_frame),2))
print("positives:", int(sep_frame["separacion_14d"].sum()))


## 6. Tiempo de lectura: target minuta


In [ ]:
minuta_columns = ", ".join(
    ["evidence_key","decision_at",*MODEL_FEATURES,"minuta_60d"]
)

def load_minuta():
    return df(f'''
        SELECT {minuta_columns}
        FROM features.lead_evidence
        WHERE minuta_60d IS NOT NULL
          AND features_refreshed_at IS NOT NULL
        ORDER BY decision_at, evidence_key
    ''')

minuta_frame = timed("load_minuta_frame", load_minuta)

print("rows:", len(minuta_frame))
print("memory_mb:", round(mem_mb(minuta_frame),2))
print("positives:", int(minuta_frame["minuta_60d"].sum()))


## 7. Tiempo de lectura: common evaluation


In [ ]:
common_columns = ", ".join(
    ["evidence_key","decision_at",*MODEL_FEATURES,"separacion_14d","minuta_60d"]
)

def load_common():
    return df(f'''
        SELECT {common_columns}
        FROM features.lead_evidence
        WHERE separacion_14d IS NOT NULL
          AND minuta_60d IS NOT NULL
          AND features_refreshed_at IS NOT NULL
        ORDER BY decision_at, evidence_key
    ''')

common_frame = timed("load_common_frame", load_common)

print("rows:", len(common_frame))
print("memory_mb:", round(mem_mb(common_frame),2))


## 8. Cardinalidad categórica


In [ ]:
categorical_profile = pd.DataFrame([
    {
        "feature": c,
        "nunique_sep": sep_frame[c].nunique(dropna=True),
        "nunique_minuta": minuta_frame[c].nunique(dropna=True),
        "null_pct_sep": sep_frame[c].isna().mean(),
        "null_pct_minuta": minuta_frame[c].isna().mean(),
    }
    for c in CATEGORICAL_FEATURES
])

categorical_profile


Una cardinalidad muy alta en `asesor`, `medio` o `canal` puede agrandar mucho el one-hot encoding y ralentizar entrenamiento.


## 9. Split temporal


In [ ]:
def do_split():
    return temporal_split(
        common_frame,
        cfg.validation_days,
        cfg.test_days
    )

split = timed("temporal_split_common", do_split)

split_summary = pd.DataFrame([
    {
        "split":"train",
        "rows":len(split.train),
        "from":split.train["decision_at"].min(),
        "to":split.train["decision_at"].max(),
    },
    {
        "split":"validation",
        "rows":len(split.validation),
        "from":split.validation["decision_at"].min(),
        "to":split.validation["decision_at"].max(),
    },
    {
        "split":"test",
        "rows":len(split.test),
        "from":split.test["decision_at"].min(),
        "to":split.test["decision_at"].max(),
    }
])

split_summary


## 10. Query exacta de scoring live: EXPLAIN


In [ ]:
score_query = f'''
SELECT
    evidence_key,
    lead_id,
    decision_at,
    documento_cliente,
    codigo_proyecto,
    asesor,
    canal,
    medio,
    {", ".join([f for f in MODEL_FEATURES if f not in {"codigo_proyecto","asesor","canal","medio"}])}
FROM features.lead_evidence e
WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
  AND features_refreshed_at IS NOT NULL
  AND NOT EXISTS (
      SELECT 1
      FROM core.fact_ciclo_comercial_unidad c
      WHERE c.documento_cliente=e.documento_cliente
        AND COALESCE(c.codigo_proyecto_ciclo,c.codigo_proyecto_unidad)=e.codigo_proyecto
        AND (
          c.fecha_separacion BETWEEN e.decision_at::date AND current_date
          OR c.fecha_venta BETWEEN e.decision_at::date AND current_date
        )
  )
ORDER BY decision_at,evidence_key
'''

plan = df("EXPLAIN " + score_query)
print("\n".join(plan.iloc[:,0].astype(str).tolist()))


## 11. Tiempo real de la query de scoring live


In [ ]:
def load_scoring_frame():
    return _read_scoring_frame(conn, cfg.score_window_days)

scoring_frame = timed(
    "read_scoring_frame",
    load_scoring_frame
)

print("rows:", len(scoring_frame))
print("memory_mb:", round(mem_mb(scoring_frame),2))
display(scoring_frame.head())


Si esta celda tarda minutos u horas, el problema está prácticamente aislado en la query `_read_scoring_frame`.


## 12. Estado del registry y serving model


In [ ]:
model_runs = df('''
SELECT
    model_run_id,
    model_version,
    trained_at,
    training_window_from,
    training_window_to,
    status,
    artifact_uri,
    metrics,
    parameters
FROM model_control.model_runs
WHERE decision_system='priorizacion_leads'
ORDER BY trained_at DESC
''')

display(model_runs.head(20))


In [ ]:
aliases = df('''
SELECT
    a.alias_name,
    a.model_run_id,
    mr.model_version,
    mr.status,
    a.updated_at
FROM model_control.model_aliases a
JOIN model_control.model_runs mr USING (model_run_id)
WHERE a.decision_system='priorizacion_leads'
ORDER BY a.updated_at DESC
''')

aliases


In [ ]:
try:
    serving = timed(
        "serving_model_lookup",
        lambda: serving_model(conn)
    )
    print(serving)
except Exception as exc:
    conn.rollback()
    serving = None
    print("NO SERVING MODEL:", repr(exc))


## 13. Artifacts disponibles


In [ ]:
artifact_root = PROJECT_ROOT / "artifacts" / "lead_scoring"

artifact_rows = []

if artifact_root.exists():
    for p in artifact_root.rglob("*.pkl"):
        artifact_rows.append({
            "path": str(p.relative_to(PROJECT_ROOT)),
            "size_mb": p.stat().st_size/(1024**2),
            "modified": datetime.fromtimestamp(p.stat().st_mtime),
        })

artifacts = pd.DataFrame(artifact_rows)

if len(artifacts):
    display(artifacts.sort_values("modified", ascending=False))
else:
    print("No se encontraron artifacts .pkl en artifacts/lead_scoring.")


## 14. Diagnóstico de tablas fuente para scoring


In [ ]:
table_stats = df('''
SELECT
    n.nspname AS schema,
    c.relname AS table,
    c.reltuples::bigint AS approx_rows,
    pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size,
    pg_total_relation_size(c.oid) AS total_bytes
FROM pg_class c
JOIN pg_namespace n ON n.oid=c.relnamespace
WHERE
    (n.nspname='features' AND c.relname='lead_evidence')
 OR (n.nspname='core' AND c.relname='fact_ciclo_comercial_unidad')
 OR (n.nspname='decision_intelligence' AND c.relname='lead_scores')
ORDER BY total_bytes DESC
''')

table_stats


## 15. Selectividad del universo live


In [ ]:
live_universe = timed(
    "live_universe_counts",
    lambda: df(f'''
        SELECT
            COUNT(*) AS total_recent,
            COUNT(*) FILTER (
                WHERE features_refreshed_at IS NOT NULL
            ) AS with_features
        FROM features.lead_evidence
        WHERE decision_at >= current_date - ({int(cfg.score_window_days)} * interval '1 day')
    ''')
)

live_universe.T


## 16. Posible cuello de botella de escritura


In [ ]:
write_estimate = pd.DataFrame([{
    "rows_to_score": len(scoring_frame),
    "estimated_records_to_build": len(scoring_frame),
    "current_scores": int(status.iloc[0]["scores"]),
    "warning": (
        "ALTO" if len(scoring_frame) > 50000
        else "MEDIO" if len(scoring_frame) > 10000
        else "BAJO"
    )
}])

write_estimate


El `scoring.py` actual crea un `records=[]`, convierte todo el DataFrame con `to_dict(orient="records")` y después usa `cursor.executemany(...)`.

Con decenas de miles de filas esto puede ser considerablemente más lento que:

- `COPY`;
- `execute_values`;
- staging table + `INSERT ... ON CONFLICT`;
- batches de 1k–5k filas.

Este notebook no modifica esa implementación: solo la cuantifica.


## 17. Timing consolidado


In [ ]:
timing_table = pd.DataFrame(TIMINGS)

if len(timing_table):
    timing_table = timing_table.sort_values(
        "seconds",
        ascending=False
    ).reset_index(drop=True)

timing_table.head(30)


## 18. Diagnóstico automático


In [ ]:
diagnostics = []

def add(severity, area, message):
    diagnostics.append({
        "severity": severity,
        "area": area,
        "message": message
    })

# Registry
if len(model_runs) == 0:
    add(
        "CRITICAL",
        "model_registry",
        "No existe ningún model_run para priorizacion_leads. El scoring no puede producir scores sin modelo."
    )

if len(aliases) == 0:
    add(
        "CRITICAL",
        "serving_alias",
        "No existe alias serving/champion. score_current_leads requiere serving_model(conn)."
    )

# Scoring query
score_timing = timing_table[
    timing_table["stage"].eq("read_scoring_frame")
]

if len(score_timing):
    secs = float(score_timing.iloc[0]["seconds"])
    if secs > 60:
        add(
            "CRITICAL",
            "scoring_sql",
            f"_read_scoring_frame tardó {secs:.1f}s. Revisar plan e índices del NOT EXISTS."
        )
    elif secs > 10:
        add(
            "WARNING",
            "scoring_sql",
            f"_read_scoring_frame tardó {secs:.1f}s; es candidato a optimización."
        )
    else:
        add(
            "OK",
            "scoring_sql",
            f"_read_scoring_frame tardó {secs:.1f}s."
        )

# Training loads
for stage in ["load_sep_frame","load_minuta_frame","load_common_frame"]:
    x = timing_table[timing_table["stage"].eq(stage)]
    if len(x) and float(x.iloc[0]["seconds"]) > 30:
        add(
            "WARNING",
            "training_sql",
            f"{stage} tardó {float(x.iloc[0]['seconds']):.1f}s."
        )

# Data volume
if len(common_frame) > 150000:
    add(
        "INFO",
        "training_volume",
        f"common_frame tiene {len(common_frame):,} filas; el entrenamiento puede ser costoso pero el volumen es suficiente."
    )

# Cardinality
for _, row in categorical_profile.iterrows():
    max_card = max(row["nunique_sep"], row["nunique_minuta"])
    if max_card > 500:
        add(
            "WARNING",
            "categorical_cardinality",
            f"{row['feature']} tiene cardinalidad {max_card}; revisar one-hot encoding."
        )

# Locks
if len(locks[locks["granted"].eq(False)]):
    add(
        "CRITICAL",
        "locks",
        "Hay locks no concedidos en PostgreSQL."
    )

# Write volume
if len(scoring_frame) > 10000:
    add(
        "WARNING",
        "score_write",
        f"Se intentarían persistir {len(scoring_frame):,} scores usando executemany; considerar batching/COPY."
    )

diagnostic_table = pd.DataFrame(diagnostics)

severity_order = pd.Categorical(
    diagnostic_table["severity"],
    categories=["CRITICAL","WARNING","INFO","OK"],
    ordered=True
)

diagnostic_table = (
    diagnostic_table
    .assign(_severity=severity_order)
    .sort_values("_severity")
    .drop(columns="_severity")
)

diagnostic_table


## 19. Diagnóstico ejecutivo


In [ ]:
print("=== LEAD SCORING PIPELINE DIAGNOSTIC ===")

if len(diagnostic_table):
    for _, row in diagnostic_table.iterrows():
        print(f"[{row['severity']}] {row['area']}: {row['message']}")

print("\n=== TOP 5 ETAPAS MÁS LENTAS ===")
for _, row in timing_table.head(5).iterrows():
    print(f"{row['stage']}: {row['seconds']:.3f}s")


## 20. Acciones sugeridas según resultado

Interpretación rápida:

```text
model_runs = 0
    → ejecutar/depurar TRAIN primero

model_runs > 0 pero aliases = 0
    → evaluar/promover modelo o crear serving provisional según el flujo existente

read_scoring_frame muy lento
    → optimizar SQL / índices de core.fact_ciclo_comercial_unidad

read_scoring_frame rápido pero score persiste lento
    → reemplazar executemany por batching / COPY / execute_values

training SQL rápido pero entrenamiento lento
    → perfilar preprocessing / one-hot / regresión logística

muchos locks
    → resolver concurrencia antes de tocar ML
```


## 21. Ejecuciones opcionales y controladas


In [ ]:
RUN_TRAIN = False
RUN_SCORE = False

print("RUN_TRAIN =", RUN_TRAIN)
print("RUN_SCORE =", RUN_SCORE)


### TRAIN opcional

Solo cambia `RUN_TRAIN=True` si quieres cronometrar realmente el entrenamiento completo.

Esto **escribe un nuevo `model_run` y artifact**.


In [ ]:
if RUN_TRAIN:
    from replica_cygnus.lead_scoring.training import train_challenger

    t0 = time.perf_counter()

    try:
        run_id, metrics = train_challenger(
            conn,
            cfg,
            PROJECT_ROOT
        )
        elapsed = time.perf_counter() - t0

        print("run_id:", run_id)
        print("elapsed_s:", round(elapsed,3))
        print(json.dumps(metrics, indent=2, default=str))
    except Exception:
        conn.rollback()
        traceback.print_exc()
else:
    print("TRAIN no ejecutado.")


### SCORE opcional

Solo cambia `RUN_SCORE=True` cuando exista un `serving_model`.

Esto **escribe `lead_scores` y `scoring_batches`**.


In [ ]:
if RUN_SCORE:
    from replica_cygnus.lead_scoring.scoring import score_current_leads

    t0 = time.perf_counter()

    try:
        result = score_current_leads(
            conn,
            cfg,
            PROJECT_ROOT
        )
        elapsed = time.perf_counter() - t0

        print("elapsed_s:", round(elapsed,3))
        print(json.dumps(result, indent=2, default=str))
    except Exception:
        conn.rollback()
        traceback.print_exc()
else:
    print("SCORE no ejecutado.")


## 22. Export opcional


In [ ]:
EXPORT = False

if EXPORT:
    out = PROJECT_ROOT / "reports" / "lead_scoring_pipeline_diagnostic"
    out.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    for name, frame in {
        "status": status,
        "activity": activity,
        "locks": locks,
        "indexes": indexes,
        "evidence_summary": evidence_summary,
        "categorical_profile": categorical_profile,
        "split_summary": split_summary,
        "table_stats": table_stats,
        "timings": timing_table,
        "diagnostics": diagnostic_table,
    }.items():
        if isinstance(frame, pd.DataFrame):
            frame.to_csv(
                out / f"{name}_{stamp}.csv",
                index=False
            )

    print("Exportado en:", out)
else:
    print("EXPORT=False")


## 23. Cierre


In [ ]:
conn.close()
print("Conexión PostgreSQL cerrada.")
